In [1]:
pip install gdown

In [2]:
# Download the zip file from Google Drive
import gdown
import os

# link: https://drive.google.com/file/d/1xNDwcGyhqY_7tdVc58YSJCS4DptDoEUL/view?usp=sharing
file_id = '1xNDwcGyhqY_7tdVc58YSJCS4DptDoEUL'
output_filename = 'archive.zip'

gdown.download(id=file_id, output=output_filename, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1xNDwcGyhqY_7tdVc58YSJCS4DptDoEUL
From (redirected): https://drive.google.com/uc?id=1xNDwcGyhqY_7tdVc58YSJCS4DptDoEUL&confirm=t&uuid=5810e357-565e-4743-97f5-afb184be7fe9
To: /content/archive.zip
100%|██████████| 1.27G/1.27G [00:17<00:00, 73.6MB/s]


'archive.zip'

In [3]:
# Extract the contents of the zip file
import zipfile

if os.path.exists(output_filename):
    with zipfile.ZipFile(output_filename, 'r') as zip_ref:
        zip_ref.extractall('.') # Extract to the current directory
    print(f"Successfully extracted '{output_filename}'")
    # Optionally, remove the zip file after extraction
    # os.remove(output_filename)
else:
    print(f"Error: The file '{output_filename}' was not found.")

Successfully extracted 'archive.zip'


In [4]:
# List the contents of the current directory to see extracted files
print("Contents of the current directory after extraction:")
print(os.listdir('.'))

Contents of the current directory after extraction:
['.config', 'model_fp32_simplified.onnx', 'mobilemodel_int8.onnx', 'test', 'archive.zip', 'train', 'sample_data']


In [5]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

transform=transforms.Compose([
    transforms.Resize((110, 100)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

full_dataset = datasets.ImageFolder(root='train', transform=transform)

In [6]:
test_dataset = datasets.ImageFolder(root='test', transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [7]:
!pip install onnxscript


In [8]:
!pip install onnxsim

In [9]:
from onnxsim import simplify
import onnx
import onnxruntime as ort

In [10]:
import psutil, os
import numpy as np

def get_ram():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024**2  # Convert bytes → MB

In [11]:
def measure_onnx_memory(model_path):
    print(f"\n===== Measuring {model_path} =====")

    # 1. Baseline
    baseline_ram = get_ram()
    print(f"Baseline RAM: {baseline_ram:.2f} MB")

    # 2. Load model
    session = ort.InferenceSession(
        model_path,
        providers=['CPUExecutionProvider']
    )
    input_name = session.get_inputs()[0].name

    after_load_ram = get_ram()
    print(f"After Load RAM: {after_load_ram:.2f} MB")

    # 3. Warmup
    dummy = np.random.randn(1, 3, 110, 100).astype(np.float32)
    for _ in range(10):
        _ = session.run(None, {input_name: dummy})

    # 4. Steady-state inference
    ram_before_inf = get_ram()

    for _ in range(100):
        _ = session.run(None, {input_name: dummy})

    ram_after_inf = get_ram()
    steady_ram = ram_after_inf

    print(f"Steady-State RAM: {steady_ram:.2f} MB")

    return {
        "baseline": baseline_ram,
        "after_load": after_load_ram,
        "steady": steady_ram
    }

RESNET

In [12]:
# fp32_results = measure_onnx_memory("model_fp32_simplified.onnx")
# int8_results = measure_onnx_memory("resnetmodel_int8.onnx")



CUSTOM MODEL 2

In [13]:
# fp32_results = measure_onnx_memory("model_fp32_simplified.onnx")
# int8_results = measure_onnx_memory("model_int8.onnx")

MOBILENET

In [14]:
# fp32_results = measure_onnx_memory("model_fp32_simplified.onnx")
int8_results = measure_onnx_memory("mobilemodel_int8.onnx")


===== Measuring mobilemodel_int8.onnx =====
Baseline RAM: 405.48 MB
After Load RAM: 420.43 MB
Steady-State RAM: 425.27 MB
